In [ ]:
!pip install -q fastparquet
!pip install -q neuralforecast

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # оставляем только первую GPU

import torch
torch.multiprocessing.set_start_method('spawn', force=True)

In [ ]:
import pandas as pd
import numpy as np
from neuralforecast import NeuralForecast
from neuralforecast.models import LSTM, GRU, NHITS

In [ ]:
# данные о реальном спросе
real_demand = pd.read_parquet('/kaggle/input/datasets/faibus/diploma/real_demand_data.parquet', engine='fastparquet')

# выгружаем типы рядов
ts_dict_df = pd.read_csv('/kaggle/input/datasets/faibus/diploma/ts_dict_df')[['SKU_id', 'ts_type']]

# данные о праздниках 
holidays = pd.read_excel('/kaggle/input/datasets/faibus/diploma/holidays.xlsx')
holidays['Holidays'] = pd.to_datetime(holidays['Holidays'], format='%Y.%m.%d')

# данные о погоде
weather = pd.read_csv('/kaggle/input/datasets/faibus/diploma/weather.csv')
weather['final_temp'] = (weather['temperature_ekb'] * 0.14) + (weather['temperature_msk'] * 0.22) + (weather['temperature_nov'] * 0.11) + (weather['temperature_ros'] * 0.16) + (weather['temperature_sam'] * 0.15) + (weather['temperature_spb'] * 0.09) + (weather['temperature_tve'] * 0.13)
weather = weather[['date', 'final_temp']]
weather['date'] = pd.to_datetime(weather['date'], format='%Y-%m-%d')

# данные о днях промо
promo_days = pd.read_parquet('/kaggle/input/datasets/faibus/diploma/promo_days.parquet', engine='fastparquet')

# данные о характеристиках SKU
sku_charasterictics = pd.read_parquet('/kaggle/input/datasets/faibus/diploma/sku_features.parquet', engine='fastparquet')[['SKU_id', 'Category4', 'Brand', 'SupplierID']]

In [ ]:
# Выделяем SKU, принадлежащие к типу lumpy
lumpy_ts = list(ts_dict_df.query(" ts_type == 'smooth' ")['SKU_id'])

# фильтруем датасет по lumpy_ts
real_demand = real_demand.query(" SKU_id.isin(@lumpy_ts) ")

# причесываем датасет
real_demand = real_demand.rename(columns = {'date':'ds', 'real_demand':'y', 'SKU_id':'unique_id'})[['unique_id', 'ds', 'y']]

real_demand

In [ ]:
# джойним характеристики SKU
real_demand = pd.merge(real_demand, sku_charasterictics, left_on = ['unique_id'], right_on = ['SKU_id'])

# создаём дни праздников (корректнее сказать "признак выходных")
real_demand['is_holiday'] = real_demand['ds'].isin(holidays['Holidays'])

# джойним промо-дни
real_demand = pd.merge(real_demand, promo_days, how = 'left', left_on = ['ds', 'unique_id'], right_on = ['date', 'SKU_id'])

# джойним погодные условия (температуру)
real_demand = pd.merge(real_demand, weather, how = 'left', left_on = ['ds'], right_on = ['date'])

# заполяем пропуски в промо-днях значением False
real_demand['is_promo'] = real_demand['is_promo'].fillna(False)

# оставляем только нужные колонки
real_demand = real_demand[['unique_id', 'ds', 'y', 'is_holiday', 'is_promo','final_temp', 'Category4', 'Brand', 'SupplierID']].copy()

# генерим временные фичи             
real_demand['month'] = real_demand['ds'].dt.month
real_demand['day'] = real_demand['ds'].dt.day
real_demand['dayofweek'] = real_demand['ds'].dt.dayofweek + 1 

# меняем тип данных bool -> int
real_demand['is_holiday'] = real_demand['is_holiday'].astype(int)
real_demand['is_promo'] = real_demand['is_promo'].astype(int)

real_demand

### Применяем Label Encoding для категориальных колонок

In [ ]:
# Кодируем в категории (обязательно для NeuralForecast)
static_cols = ['Category4', 'Brand', 'SupplierID']
for col in static_cols:
    real_demand[col] = real_demand[col].astype('category').cat.codes

In [ ]:
real_demand

In [ ]:
horizon = 14

# Задаем экзогенные переменные
exog_cols = ['is_holiday', 'is_promo', 'final_temp', 'month', 'day', 'dayofweek']
static_cols = ['Category4', 'Brand', 'SupplierID']

# Общие параметры для всех моделей (распаковываем)
trainer_params = {
    'accelerator': 'gpu',
    'devices': 1,
    'strategy': 'auto' 
}

common = dict(
    h=horizon,
    scaler_type='robust',
    futr_exog_list=exog_cols,
    stat_exog_list=static_cols,
    **trainer_params
)

# 1. Создаем список с моделями
models = [
    # === LSTM ===
    LSTM(**common, input_size=21, max_steps=1000, alias='LSTM_in21_s1000'),
    
    # === GRU ===
    GRU(**common, input_size=21, max_steps=1000, alias='GRU_in21_s1000'),
    
    # === NHITS ===
    NHITS(**common, input_size=35, max_steps=1000,
          n_freq_downsample=[7, 1, 1],
          dropout_prob_theta=0.1,
          alias='NHITS_drop01'),
]

In [ ]:
# задаем параметры
step_size = 7
n_windows = 5

# определяем минимально необходимую длину ряда
min_required = n_windows * step_size + horizon

# Группировка и подсчёт длины
series_len = real_demand.groupby('unique_id').size()
long_series = series_len[series_len >= min_required].index

# Оставляем только длинные ряды
real_demand_filtered = real_demand[real_demand['unique_id'].isin(long_series)]

# Создаём static_df (один ряд на unique_id)
static_df = real_demand_filtered[['unique_id'] + static_cols].drop_duplicates(subset='unique_id')

# Создаём экземпляр NeuralForecast со списком моделей
nf = NeuralForecast(models=models, freq='D')

# Теперь запускаем кросс-валидацию на отфильтрованных данных
cv_results = nf.cross_validation(
    df=real_demand_filtered,
    static_df=static_df,
    h=horizon,
    step_size=step_size,
    n_windows=n_windows
)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# -------------------------------
# 1. Определение метрик
# -------------------------------

def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype='float64')
    y_pred = np.asarray(y_pred, dtype='float64')
    denominator = np.abs(y_true) + np.abs(y_pred)
    mask = denominator > 0
    if mask.sum() == 0:
        return np.nan
    return (2 * np.abs(y_true[mask] - y_pred[mask]) / denominator[mask]).mean() * 100

def wape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype='float64')
    y_pred = np.asarray(y_pred, dtype='float64')
    denominator = np.abs(y_true).sum()
    if denominator == 0:
        return np.nan
    return np.abs(y_true - y_pred).sum() / denominator * 100

# -------------------------------
# 2. Функция для вычисления метрик по каждому окну (без unique_id)
# -------------------------------

def compute_metrics_per_window(cv_results, model_columns, metrics_dict):
    """
    cv_results: DataFrame от sf.cross_validation()
    model_columns: список имён столбцов с прогнозами моделей
    metrics_dict: словарь вида {'mae': mae, 'rmse': rmse, ...}
    Возвращает DataFrame с колонками: cutoff, model, и каждой метрикой.
    Для каждого cutoff (окна) метрика вычисляется по всем точкам всех unique_id.
    """
    records = []
    grouped = cv_results.groupby('cutoff')   # группируем только по окну
    
    for cutoff, group in grouped:
        y_true = group['y'].values
        for model in model_columns:
            y_pred = group[model].values
            row = {'cutoff': cutoff, 'model': model}
            for metric_name, metric_func in metrics_dict.items():
                row[metric_name] = metric_func(y_true, y_pred)
            records.append(row)
    
    return pd.DataFrame(records)

# -------------------------------
# 3. Применение к вашим данным
# -------------------------------

# Предполагаем, что cv_results уже получен после cross_validation
# Определяем столбцы моделей (все, кроме служебных)
model_columns = [col for col in cv_results.columns 
                 if col not in ['unique_id', 'ds', 'cutoff', 'y']]

metrics = {
    'mae': mae,
    'rmse': rmse,
    'smape': smape,
    'wape': wape
}

# Вычисляем метрики для каждого окна (без разбивки по unique_id)
metrics_per_window = compute_metrics_per_window(cv_results, model_columns, metrics)

# -------------------------------
# 4. Агрегация результатов
# -------------------------------

# 4.1. Среднее по всем окнам для каждой модели
summary_mean = metrics_per_window.groupby('model')[['mae', 'rmse', 'smape', 'wape']].mean()


# 4.2. Детальная статистика (mean, std, min, max) по окнам
summary_stats = metrics_per_window.groupby('model')[['mae', 'rmse', 'smape', 'wape']].agg(['mean', 'std', 'min', 'max'])


summary_mean

In [ ]:
# -------------------------------
# 5. Метрики по сегментам объёма SKU
# -------------------------------

# 5.1. Средний спрос по каждому ряду
avg_demand = real_demand_filtered.groupby('unique_id')['y'].mean()

# 5.2. Разбиваем на сегменты
segments = {
    'High volume (top 25%)': avg_demand[avg_demand >= avg_demand.quantile(0.75)].index,
    'Mid volume (25-75%)': avg_demand[(avg_demand >= avg_demand.quantile(0.25)) & 
                                      (avg_demand < avg_demand.quantile(0.75))].index,
    'Low volume (bottom 25%)': avg_demand[avg_demand < avg_demand.quantile(0.25)].index,
}

# 5.3. Считаем метрики для каждого сегмента
segment_results = []

for segment_name, segment_ids in segments.items():
    mask = cv_results['unique_id'].isin(segment_ids)
    cv_segment = cv_results[mask]
    
    segment_metrics = compute_metrics_per_window(cv_segment, model_columns, metrics)
    segment_summary = segment_metrics.groupby('model')[['mae', 'rmse', 'smape', 'wape']].mean()
    segment_summary['segment'] = segment_name
    segment_summary['n_series'] = len(segment_ids)
    segment_results.append(segment_summary)

summary_by_segment = pd.concat(segment_results)
summary_by_segment = summary_by_segment.reset_index().set_index(['segment', 'n_series', 'model'])
summary_by_segment = summary_by_segment.sort_index()

summary_by_segment